In [ ]:
import pandas as pd
import numpy as np
import re, os, joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score

%matplotlib inline
plt.rcParams['figure.dpi'] = 150

In [ ]:
data_path = os.path.join('data', 'spam.csv')
df = pd.read_csv(data_path)
print(f"Dataset: {df.shape[0]} messages")
df['label'].value_counts()

In [ ]:
df.head(10)

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)
df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=10000)),
    ('clf', MultinomialNB(alpha=0.1))
])
nb_pipeline.fit(X_train, y_train)
nb_preds = nb_pipeline.predict(X_test)
nb_f1 = f1_score(y_test, nb_preds, pos_label='spam')

print("=== MultinomialNB ===")
print(classification_report(y_test, nb_preds))
print(f"Spam F1: {nb_f1:.4f}")

In [ ]:
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=10000)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0))
])
lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)
lr_f1 = f1_score(y_test, lr_preds, pos_label='spam')

print("=== LogisticRegression ===")
print(classification_report(y_test, lr_preds))
print(f"Spam F1: {lr_f1:.4f}")

In [ ]:
if nb_f1 >= lr_f1:
    best_name, best_pipeline, best_preds, best_f1 = "MultinomialNB", nb_pipeline, nb_preds, nb_f1
else:
    best_name, best_pipeline, best_preds, best_f1 = "LogisticRegression", lr_pipeline, lr_preds, lr_f1

print(f"Best model: {best_name} (Spam F1 = {best_f1:.4f})")
assert best_f1 > 0.90, f"Spam F1 {best_f1:.4f} is below 0.90 threshold" 

In [ ]:
results_dir = os.path.join('model', 'results')
model_path = os.path.join('model', 'model.pkl')
os.makedirs(results_dir, exist_ok=True)
joblib.dump(best_pipeline, model_path)
print(f"Model saved to {model_path}")

In [ ]:
cm = confusion_matrix(y_test, best_preds, labels=['ham', 'spam'])
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'],
            yticklabels=['Ham', 'Spam'], ax=ax, linewidths=0.5,
            annot_kws={'size': 16, 'weight': 'bold'})
ax.set_xlabel('Predicted', fontsize=13, fontweight='bold')
ax.set_ylabel('Actual', fontsize=13, fontweight='bold')
ax.set_title(f'Confusion Matrix - {best_name}', fontsize=15, fontweight='bold', pad=12)
plt.tight_layout()
fig.savefig(os.path.join(results_dir, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
report = classification_report(y_test, best_preds, output_dict=True)
metrics_df = pd.DataFrame(report).T
metrics_df = metrics_df.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
metrics_df = metrics_df[['precision', 'recall', 'f1-score']]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(metrics_df.index))
width = 0.25
colors = ['#457B9D', '#2D6A4F', '#E9C46A']
labels = ['Precision', 'Recall', 'F1-Score']

for i, (col, color, label) in enumerate(zip(metrics_df.columns, colors, labels)):
    bars = ax.bar(x + i * width, metrics_df[col], width, label=label, color=color, edgecolor='white', linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_df.index.str.title(), fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title(f'Classification Report - {best_name}', fontsize=14, fontweight='bold', pad=12)
ax.legend(fontsize=10, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(os.path.join(results_dir, 'classification_report.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
models = ['MultinomialNB', 'LogisticRegression']
f1_scores = [nb_f1, lr_f1]
colors = ['#E63946' if f == max(f1_scores) else '#457B9D' for f in f1_scores]
bars = ax.barh(models, f1_scores, color=colors, edgecolor='white', height=0.5)
for bar, f1 in zip(bars, f1_scores):
    ax.text(bar.get_width() - 0.03, bar.get_y() + bar.get_height()/2, f'{f1:.4f}',
            ha='right', va='center', fontsize=13, fontweight='bold', color='white')
ax.set_xlim(0, 1.05)
ax.set_xlabel('Spam Class F1 Score', fontsize=12, fontweight='bold')
ax.set_title('Model Comparison', fontsize=14, fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(os.path.join(results_dir, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
test_messages = [
    "Congratulations! You've won a free iPhone! Click here to claim now!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account has been compromised. Call 0800-123-456 immediately!",
    "Can you pick up some milk on your way home?"
]

for msg in test_messages:
    cleaned = clean_text(msg)
    pred = best_pipeline.predict([cleaned])[0]
    proba = best_pipeline.predict_proba([cleaned])[0]
    conf = max(proba) * 100
    print(f"[{pred.upper():4s}] ({conf:.1f}%) {msg[:70]}")